In [ ]:
# =========================================================
# AI DRIVER MONITORING SYSTEM
# =========================================================
# FEATURES
# =========================================================
# 1. Drowsiness Detection
# 2. Yawning Detection
# 3. Real Mobile Phone Detection (YOLOv8)
# 4. Webcam / Video File Support
# 5. Live Alarm Alerts
# 6. Face Detection Box
# 7. FPS Display
# 8. Better Stability
# =========================================================

# =========================================================
# IMPORT LIBRARIES
# =========================================================

from scipy.spatial import distance
from imutils import face_utils
from ultralytics import YOLO
from pygame import mixer

import numpy as np
import imutils
import dlib
import cv2
import time

# =========================================================
# INITIALIZE SOUND
# =========================================================

mixer.init()

mixer.music.load("music.wav")

# =========================================================
# LOAD YOLO MODEL
# =========================================================

# DOWNLOAD yolov8n.pt OR yolov8s.pt
# PUT INSIDE PROJECT FOLDER

model = YOLO(
    r"D:\my projects\drowsiness final\dds\yolov8n.pt"
)

# =========================================================
# EYE ASPECT RATIO FUNCTION
# =========================================================

def eye_aspect_ratio(eye):

    A = distance.euclidean(eye[1], eye[5])

    B = distance.euclidean(eye[2], eye[4])

    C = distance.euclidean(eye[0], eye[3])

    ear = (A + B) / (2.0 * C)

    return ear


# =========================================================
# MOUTH ASPECT RATIO FUNCTION
# =========================================================

def mouth_aspect_ratio(mouth):

    A = distance.euclidean(mouth[2], mouth[10])

    B = distance.euclidean(mouth[4], mouth[8])

    C = distance.euclidean(mouth[0], mouth[6])

    mar = (A + B) / (2.0 * C)

    return mar


# =========================================================
# THRESHOLDS
# =========================================================

EYE_THRESH = 0.30

MOUTH_THRESH = 0.75

FRAME_CHECK = 12

# =========================================================
# LOAD DLIB DETECTOR
# =========================================================

detect = dlib.get_frontal_face_detector()

predict = dlib.shape_predictor(
    r"D:\my projects\drowsiness final\shape_predictor_68_face_landmarks.dat"
)

# =========================================================
# FACIAL LANDMARKS
# =========================================================

(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_68_IDXS["left_eye"]

(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_68_IDXS["right_eye"]

(mStart, mEnd) = (48, 68)

# =========================================================

if not cap.isOpened():

    print("Cannot Open Camera / Video")

    exit()

# =========================================================
# VARIABLES
# =========================================================

flag = 0

yawn_start = None

prev_time = 0

# =========================================================
# MAIN LOOP
# =========================================================

while True:

    ret, frame = cap.read()

    if not ret:

        print("Cannot Read Frame")

        break

    # =====================================================
    # PREPROCESSING
    # =====================================================

    frame = cv2.flip(frame, 1)

    frame = imutils.resize(frame, width=900)

    # BRIGHTNESS IMPROVEMENT

    frame = cv2.convertScaleAbs(
        frame,
        alpha=1.2,
        beta=20
    )

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # =====================================================
    # FPS CALCULATION
    # =====================================================

    current_time = time.time()

    fps = 1 / (current_time - prev_time)

    prev_time = current_time

    # =====================================================
    # FACE DETECTION
    # =====================================================

    subjects = detect(gray, 1)

    for subject in subjects:

        # =================================================
        # FACE RECTANGLE
        # =================================================

        x1 = subject.left()

        y1 = subject.top()

        x2 = subject.right()

        y2 = subject.bottom()

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 0),
            2
        )

        # =================================================
        # LANDMARK DETECTION
        # =================================================

        shape = predict(gray, subject)

        shape = face_utils.shape_to_np(shape)

        # =================================================
        # EYES
        # =================================================

        leftEye = shape[lStart:lEnd]

        rightEye = shape[rStart:rEnd]

        # =================================================
        # MOUTH
        # =================================================

        mouth = shape[mStart:mEnd]

        # =================================================
        # EAR CALCULATION
        # =================================================

        leftEAR = eye_aspect_ratio(leftEye)

        rightEAR = eye_aspect_ratio(rightEye)

        ear = (leftEAR + rightEAR) / 2.0

        # =================================================
        # MAR CALCULATION
        # =================================================

        mar = mouth_aspect_ratio(mouth)

        # =================================================
        # DRAW EYES
        # =================================================

        leftEyeHull = cv2.convexHull(leftEye)

        rightEyeHull = cv2.convexHull(rightEye)

        cv2.drawContours(
            frame,
            [leftEyeHull],
            -1,
            (0, 255, 0),
            1
        )

        cv2.drawContours(
            frame,
            [rightEyeHull],
            -1,
            (0, 255, 0),
            1
        )

        # =================================================
        # DRAW MOUTH
        # =================================================

        mouthHull = cv2.convexHull(mouth)

        cv2.drawContours(
            frame,
            [mouthHull],
            -1,
            (255, 0, 0),
            1
        )

        # =================================================
        # DISPLAY VALUES
        # =================================================

        cv2.putText(
            frame,
            f"EAR: {ear:.2f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"MAR: {mar:.2f}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 0, 0),
            2
        )

        # =================================================
        # DROWSINESS DETECTION
        # =================================================

        if ear < EYE_THRESH:

            flag += 1

            cv2.putText(
                frame,
                "EYES CLOSED",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

            if flag >= FRAME_CHECK:

                cv2.putText(
                    frame,
                    "DROWSINESS ALERT!",
                    (240, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3
                )

                if not mixer.music.get_busy():

                    mixer.music.play()

        else:

            flag = 0

        # =================================================
        # YAWNING DETECTION
        # =================================================

        if mar > MOUTH_THRESH:

            cv2.putText(
                frame,
                "YAWNING...",
                (20, 160),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )

            if yawn_start is None:

                yawn_start = time.time()

            yawn_duration = time.time() - yawn_start

            cv2.putText(
                frame,
                f"YAWN TIME: {int(yawn_duration)} sec",
                (20, 200),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 0),
                2
            )

            if yawn_duration >= 3:

                cv2.putText(
                    frame,
                    "YAWNING ALERT!",
                    (240, 110),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3
                )

                if not mixer.music.get_busy():

                    mixer.music.play()

        else:

            yawn_start = None

    # =====================================================
    # YOLO OBJECT DETECTION
    # =====================================================

    results = model(frame, verbose=False)

    for result in results:

        boxes = result.boxes

        for box in boxes:

            cls = int(box.cls[0])

            label = model.names[cls]

            confidence = float(box.conf[0])

            # =================================================
            # PHONE DETECTION
            # =================================================

            if label == "cell phone" and confidence > 0.35:

                px1, py1, px2, py2 = map(
                    int,
                    box.xyxy[0]
                )

                # RED RECTANGLE

                cv2.rectangle(
                    frame,
                    (px1, py1),
                    (px2, py2),
                    (0, 0, 255),
                    3
                )

                # TEXT

                cv2.putText(
                    frame,
                    "MOBILE PHONE DETECTED",
                    (px1, py1 - 15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

                cv2.putText(
                    frame,
                    f"{confidence:.2f}",
                    (px1, py2 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    2
                )

                # ALARM

                if not mixer.music.get_busy():

                    mixer.music.play()

    # =====================================================
    # DISPLAY FPS
    # =====================================================

    cv2.putText(
        frame,
        f"FPS: {int(fps)}",
        (760, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    # =====================================================
    # TITLE
    # =====================================================

    cv2.putText(
        frame,
        "AI DRIVER MONITORING SYSTEM",
        (180, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2
    )

    # =====================================================
    # SHOW WINDOW
    # =====================================================

    cv2.imshow(
        "AI Driver Monitoring System",
        frame
    )

    # =====================================================
    # EXIT
    # =====================================================

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):

        break

# =========================================================
# CLEANUP
# =========================================================

cap.release()

cv2.destroyAllWindows()